# AI Iridology: Stage 2 Continuous Glucose Regression

This notebook replicates the **EasyGlucose** architectural approach for non-invasive continuous glucose monitoring using images of the human iris.

### Pipeline Overview:
1. **Topographical Extraction:** Simulates Daugman's rubber sheet model to extract the specific iris sector (e.g., Pancreas).
2. **Regression CNN:** Uses a Deep Convolutional Neural Network with a `Linear` activation function to output a continuous scalar value (mg/dL).
3. **Loss Function:** Optimized using Mean Absolute Error (MAE).
4. **Clinical Evaluation:** Validated using the Pearson Correlation Coefficient and the Clarke Error Grid.

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

print("TensorFlow Version:", tf.__version__)

## 1. Topographical ROI Extraction (Daugman Simulation)

In [ ]:
def extract_topographical_roi(image):
    """
    Simulates Daugman's rubber sheet model to unwrap the iris 
    and extract the specific sector corresponding to the pancreas.
    """
    h, w = image.shape[:2]
    # Simple simulated sector crop (bottom-right quadrant)
    sector = image[h//2:h, w//2:w]
    return cv2.resize(sector, (150, 150))

## 2. Data Loading & Preprocessing
*(Note: Currently points to the synthetic dataset used to validate the architecture)*

In [ ]:
def load_and_preprocess_data(base_dir="synthetic_iris_dataset"):
    csv_path = os.path.join(base_dir, "glucose_labels.csv")
    images_dir = os.path.join(base_dir, "images")
    
    if not os.path.exists(csv_path):
        print("Dataset not found! Please run generate_mock_dataset.py first.")
        return None, None, None, None

    df = pd.read_csv(csv_path)
    X, y = [], []
    
    for index, row in df.iterrows():
        img_path = os.path.join(images_dir, row["image_filename"])
        img = cv2.imread(img_path)
        if img is not None:
            roi = extract_topographical_roi(img)
            roi = roi.astype('float32') / 255.0
            X.append(roi)
            y.append(row["glucose_mg_dl"])
            
    return train_test_split(np.array(X), np.array(y), test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = load_and_preprocess_data()
if X_train is not None:
    print(f"Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

## 3. Deep Regression Architecture
Notice the **Linear** activation function and **Mean Absolute Error (MAE)** loss.

In [ ]:
def build_regression_model(input_shape=(150, 150, 3)):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='linear') # Linear for continuous regression
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mae', metrics=['mse'])
    return model

model = build_regression_model()
model.summary()

In [ ]:
if X_train is not None:
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=15,
        batch_size=8,
        verbose=1
    )

## 4. Clinical Evaluation (Clarke Error Grid & Pearson Correlation)
Evaluation metrics are measured using real clinical standards.

In [ ]:
if X_train is not None:
    y_pred = model.predict(X_test).flatten()
    
    corr, _ = pearsonr(y_test, y_pred)
    print(f"\nPearson Correlation Coefficient: {corr:.3f}")
    
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test, y_pred, alpha=0.7, color='blue')
    plt.plot([50, 300], [50, 300], 'k--', label='Perfect Agreement')
    plt.plot([50, 300], [60, 360], 'g--', label='+20% Margin')
    plt.plot([50, 300], [40, 240], 'g--')
    plt.title("Clinical Benchmark: Clarke Error Grid")
    plt.xlabel("Reference Glucose (mg/dL)")
    plt.ylabel("Predicted Glucose (mg/dL)")
    plt.xlim(50, 300)
    plt.ylim(50, 300)
    plt.grid(True)
    plt.legend()
    plt.show()